# Time-explicit LCA of a Danish *sommerhus*

<div style="display: flex; justify-content: center; width: 600px; margin: auto;">
  <img src="https://www.dansk-sommerhusferie.dk/UrlaubTheme/Styles/img/Dansksommerhusferie.jpg" style="border-radius: 15px;">
</div>

Built of wood, heated with a heat pump, lived in for 50 years.

```mermaid
flowchart LR
    tree(🌲 timber tree):::fg-->timber
    timber(🪵 structural timber production):::fg-->construction
    glass_wool(🧵 market for glass wool mat):::ei-->construction
    construction(🏗️ sommerhus construction):::fg-->living
    heat_pump(🔥 heat pump, brine-water 10kW):::ei-->living
    grid(⚡ market group for electricity, low voltage):::ei-->living
    living(🏠 living in the sommerhus):::fg-->waste_wood(🔥 incineration of waste wood):::fg
    living-->fu(FU: 50 years of habitation)

    classDef ei color:#222832, fill:#3fb1c5, stroke:none;
    classDef fg color:#222832, fill:#9c5ffd, stroke:none;
```

<span style="color:#9c5ffd">■</span> foreground &nbsp;&nbsp;
<span style="color:#3fb1c5">■</span> background (premise vintages 2020 / 2030 / 2040 / 2050)

## 0 | The product system

We make the following assumptions:


In [ ]:
LIFETIME = 50  # years, EN 15978 reference service life for a building
TIMBER_VOLUME = 12  # m3 of structural timber in the frame
INSULATION_MASS = 1200  # kg of glass wool
HEAT_PUMPS = 3  # bought over the lifetime: one plus two replacements
ELECTRICITY_PER_YEAR = 1400  # kWh/a: hot water (heat-pump-assisted) plus household

BG_DATABASE = "ei_cutoff_3.12_remind-eu_SSP2-NDC_2020"
METHOD = ("IPCC 2021", "climate change", "GWP 100a, incl. H and bio CO2")

We've prepared the code to set up the sommerhus system with all its nodes and edges in [`sommerhus_system.py`](sommerhus_system.py) to save some time. Have a look if you want to see
how - but it's just normal brightway stuff.

In [ ]:
from sommerhus_system import build_system

foreground = build_system(
    lifetime=LIFETIME,
    timber_volume=TIMBER_VOLUME,
    insulation_mass=INSULATION_MASS,
    heat_pumps=HEAT_PUMPS,
    electricity_per_year=ELECTRICITY_PER_YEAR,
    background_database=BG_DATABASE,
    method=METHOD,
)
[node["name"] for node in foreground]

### The functional unit, and a static LCA to compare against

50 years of living in the *sommerhus*:


In [ ]:
import bw2calc as bc
import bw2data as bd

living = bd.get_node(database="foreground", name="living in the sommerhus")

static_lca = bc.LCA({living: 1}, METHOD)
static_lca.lci()
static_lca.lcia()

print(f"static score: {static_lca.score:,.0f} kg CO2-eq")

## 1 | When does what happen?

A `TemporalDistribution` says what **share** of an exchange happens **when**, relative to the
process consuming it.

Three of them are given below as examples: the **timber delivery**, the **tree's CO2 uptake**
and the **electricity use**. Two more are yours to write, further down.


In [ ]:
import numpy as np
from bw_timex import TemporalDistribution, easy_timedelta_distribution

# The timber and the insulation arrive over the last nine months before move-in.
# We create this one directly from the TemporalDistribution class, specifying date and amount arrays.
td_construction = TemporalDistribution(
    date=np.array([-9, -6, -2, -1], dtype="timedelta64[M]"),  # months before the consumer
    amount=np.array([0.2, 0.3, 0.3, 0.2]),                    # shares, summing to 1
)

# The tree grows for 40 years before it is felled. Its uptake *rate* is bell-shaped (slow
# start, fast middle, tapering off), which makes the carbon stock an S-curve.
# Here, we use the easy_timedelta_distribution function that helps with more complex setups.
td_biogenic_uptake = easy_timedelta_distribution(
    start=-40, end=0, resolution="Y", steps=41, kind="normal", param=0.2
)

# Electricity is used evenly over the years of habitation.
td_electricity = easy_timedelta_distribution(
    start=0, end=LIFETIME, resolution="Y", steps=LIFETIME + 1, kind="uniform"
)

Let's have a look:

In [ ]:
td_construction.graph(resolution="M")

In [ ]:
td_biogenic_uptake.graph(resolution="Y")

Along a supply chain, the TDs of consecutive exchanges are **convolved** together: the dates are shifted against one another, the amounts multiply. Have a look, and try to grasp what's going on. You can also create some TDs yourself to test if your intuition is right.

In [ ]:
(td_construction * td_biogenic_uptake).graph(resolution="M")

A TD belongs on an **exchange**.
[`add_temporal_distribution_to_exchange`](https://docs.brightway.dev/projects/bw-timex/en/latest/content/api/bw_timex/utils/index.html)
finds the exchange for you - give it enough to identify producer and consumer unambiguously:


In [ ]:
from bw_timex.utils import add_temporal_distribution_to_exchange

add_temporal_distribution_to_exchange(
    td_construction,
    input_name="structural timber production, carbon split",
    output_name="sommerhus construction, wood",
)

add_temporal_distribution_to_exchange(
    td_construction,
    input_name="market for glass wool mat",  # a background node, so pin it down further
    input_location="GLO",
    input_database=BG_DATABASE,
    output_name="sommerhus construction, wood",
)

add_temporal_distribution_to_exchange(
    td_biogenic_uptake,
    input_name="Carbon dioxide, in air",  # a biosphere flow works just the same
    output_name="timber tree",
)

add_temporal_distribution_to_exchange(
    td_electricity,
    input_name="market group for electricity, low voltage",
    input_location="GLO",
    input_database=BG_DATABASE,
    output_name="living in the sommerhus",
)

### 🛠️ Your turn: the last two

Two exchanges in the product system above still carry to TD. If we leave it like that, `bw_timex` will assume they are instant - but we don't want that. Come up with TDs for the remaining exchanges and attach them.

> The heat pump is a background node (`"heat pump production, brine-water, 10kW"`, location `"RoW"`), the incineration is in the foreground.


In [ ]:
# TODO: write td_heat_pumps and td_eol (see the table above) and attach them with
#       add_temporal_distribution_to_exchange(...)

In [ ]:
# Stuck, or out of time? Uncomment the line below, run this cell twice,
# and you're back in sync with everyone else.
# %load solutions/3_sommerhus/1_temporal_distributions.py

In [ ]:
# CHECKPOINT - all six exchanges should carry a temporal distribution now
from bw_timex.utils import get_exchange

CONSTRUCTION = "sommerhus construction, wood"
LIVING = "living in the sommerhus"

for label, kwargs in [
    ("timber -> construction",
     dict(input_name="structural timber production, carbon split", output_name=CONSTRUCTION)),
    ("insulation -> construction",
     dict(input_name="market for glass wool mat", input_location="GLO",
          input_database=BG_DATABASE, output_name=CONSTRUCTION)),
    ("CO2 uptake of the tree",
     dict(input_name="Carbon dioxide, in air", output_name="timber tree")),
    ("electricity -> living",
     dict(input_name="market group for electricity, low voltage", input_location="GLO",
          input_database=BG_DATABASE, output_name=LIVING)),
    ("heat pumps -> living",
     dict(input_name="heat pump production, brine-water, 10kW", input_location="RoW",
          input_database=BG_DATABASE, output_name=LIVING)),
    ("living -> incineration",
     dict(input_name="incineration of waste wood", output_name=LIVING)),
]:
    exchange = get_exchange(**kwargs)
    print(f"{label:<28} {'ok' if 'temporal_distribution' in exchange else 'MISSING'}")

### Amounts that change over time

A TD is only about distributing *when* an exchange happens. By itself, it doesn't change the (total) amount of that exchange. For that there is **temporal
evolution factors**: factors at given dates, linearly interpolated in between, held constant outside.

For our *sommerhus*, we assume that we, as habitants, reduce our electricity demand by 1% of the original demand per year.

It goes on the very exchange that already carries the electricity TD - *when* an exchange
happens and *how much* of it happens are two separate pieces of information on one edge.


In [ ]:
from datetime import datetime

from bw_timex.utils import add_temporal_evolution_to_exchange

add_temporal_evolution_to_exchange(
    temporal_evolution_factors={
        datetime(2025, 1, 1): 1.0,
        datetime(2075, 1, 1): 0.5,  # -1% of the 2025 demand per year after interpolation
    },
    input_name="market group for electricity, low voltage",
    input_location="GLO",
    input_database=BG_DATABASE,
    output_name="living in the sommerhus",
)

The factor is looked up at each point in time the exchange occurs. This is what it
looks like over the lifetime of the house:


In [ ]:
from bw_timex import get_temporal_evolution_factor

factors = {datetime(2025, 1, 1): 1.0, datetime(2075, 1, 1): 0.5}
years = range(2020, 2091)

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 3))
plt.plot(list(years), [get_temporal_evolution_factor(factors, datetime(y, 1, 1)) for y in years])
plt.ylabel("factor on the grid draw")
plt.xlabel("year")
plt.ylim(0, 1.1)
plt.tight_layout()
plt.show()

## 2 | The background, in several vintages

So far everything referred to one background database, the 2020 vintage. But the project holds
four, built with [premise](https://github.com/polca/premise) from the **REMIND-EU SSP2-NDC**
scenario on top of **ecoinvent 3.12 cutoff**: 2020, 2030, 2040 and 2050.


In [ ]:
[name for name in bd.databases if name.startswith("ei_cutoff")]

What makes them usable for us is their metadata. premise writes the point in time each
database represents into the database itself:


In [ ]:
dict(bd.databases[BG_DATABASE])

`representative_time` is the key part. `TimexLCA` reads it by itself, so we never have
to tell it which database is which year - and for every process it places in time, it sources
from the vintage(s) nearest to that moment. (If your databases do not carry the metadata, you
can set it with `bw_timex.set_database_metadata`, or pass `database_dates` to `TimexLCA`.)


In [ ]:
for name in sorted(n for n in bd.databases if n.startswith("ei_cutoff")):
    print(f"{name:<45} {bd.databases[name]['representative_time']}")

## 3 | The timeline

Which process happens when, and which background vintage(s) it is sourced from
(`temporal_market_shares`).


In [ ]:
from bw_timex import TimexLCA

tlca = TimexLCA({living: 1}, METHOD)
tlca.build_timeline(starting_datetime="2025-01-01", temporal_grouping="month")

In [ ]:
electricity_rows = tlca.timeline[
    tlca.timeline["producer_name"] == "market group for electricity, low voltage"
]
electricity_rows[
    ["date_producer", "amount", "temporal_evolution_factor", "temporal_market_shares"]
].head(10)

### 🔍 Explore

1. How many electricity rows are there, and what differs between them?
2. Pick a row in the 2040s and look at its `temporal_market_shares`. Why is it split over two
   databases - and what would a row in 2030 look like?
3. Find the earliest row in the whole timeline. Which process is it, and why that one?
4. In `.build_timeline`, we implicitly use the default setting graph_traversal="priority". Test the other option, have a look at the computation time, and think about the differences.


## 4 | Time-explicit inventory and score

In [ ]:
tlca.lci()
tlca.static_lcia()

print(f"static:        {static_lca.score:,.0f} kg CO2-eq")
print(f"time-explicit: {tlca.static_score:,.0f} kg CO2-eq")

## 5 | Dynamic characterization

The inventory still knows *when* each emission happens, so it can be characterized
dynamically instead of with one static factor for everything.


In [ ]:
tlca.dynamic_lcia(metric="GWP", time_horizon=100)
print(f"dynamic GWP100: {tlca.dynamic_score:,.0f} kg CO2-eq")

In [ ]:
from bw_timex.utils import plot_characterized_inventory_as_waterfall

# one bar per year over a century is a lot of labels, so label every fifth one
plot_characterized_inventory_as_waterfall(tlca, xtick_interval=5)

*How that characterization works, and what else it can do, is the next notebook.*

## 6 | Same house, different decade

`TimexLCASettings` bundles one run, `TimexLCA.compare()` runs a list of them. Here: the same
*sommerhus*, moved into in 2025, 2035 and 2045.


In [ ]:
from dataclasses import replace

from bw_timex import TimexLCASettings

settings_2025 = TimexLCASettings(
    demand={living: 1},
    method=METHOD,
    timeline={"starting_datetime": "2025-01-01", "temporal_grouping": "month"},
    lcia={"metric": "GWP", "time_horizon": 100},
    label="move in 2025",
)

comparison = TimexLCA.compare(
    [
        settings_2025,
        replace(settings_2025, starting_datetime="2035-01-01", label="move in 2035"),
        replace(settings_2025, starting_datetime="2045-01-01", label="move in 2045"),
    ]
)
comparison.summary[["label", "dynamic_score"]]

## Time for a break, mh? ☕